In [1]:
import sys; sys.path.append('..')  # shared utils.py lives in the repo root
import torch
import numpy as np
import pandas as pd
import pyro
import pyro.distributions as dist
from pyro import poutine
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam
from scipy.stats import t as scipy_t
from torch.utils.data import DataLoader, TensorDataset
from dmm_features import DMMFeatures
from utils import log_returns, rogers_satchell_rv

c:\Users\mikah\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load all available series into a single pool
# add new series here when you want to experiment with additional features
spy = pd.read_csv('../data/spy_ohlcv.csv', index_col=0, parse_dates=True, skiprows=[1, 2])

pool = {
    'ret':    log_returns(spy['Close']),
    'rv':     rogers_satchell_rv(spy).clip(lower=1e-8),
    'VIX':    pd.read_csv('../data/fred_vix.csv',    index_col=0, parse_dates=True).squeeze(),
    'T10Y3M': pd.read_csv('../data/fred_t10y3m.csv', index_col=0, parse_dates=True).squeeze(),
    'BAA10Y': pd.read_csv('../data/fred_baa10y.csv', index_col=0, parse_dates=True).squeeze(),
    'DGS2':   pd.read_csv('../data/fred_dgs2.csv',   index_col=0, parse_dates=True).squeeze(),
}

for name, s in pool.items():
    valid = s.dropna()
    print(f"  {name:<8} {valid.index[0].date()} → {valid.index[-1].date()}  ({len(valid)} rows)")

  ret      1993-02-01 → 2026-05-15  (8380 rows)
  rv       1993-01-29 → 2026-05-15  (8381 rows)
  VIX      1990-01-02 → 2026-05-14  (9186 rows)
  T10Y3M   1982-01-04 → 2026-05-15  (11095 rows)
  BAA10Y   1986-01-02 → 2026-05-14  (10092 rows)
  DGS2     1976-06-01 → 2026-05-14  (12485 rows)


In [3]:
# ── FEATURE CONFIG ────────────────────────────────────────────────────────────
# This is the single place to change which features are used.
# Each entry: (name_in_pool, emission_type)
#   emission types: 'studentt'  — for returns (support: all reals)
#                   'lognormal' — for strictly positive series (rv, VIX, spreads)
#                   'normal'    — for real-valued series (yield levels/spreads)
#
# Column order here determines x layout passed to DMMFeatures.
# DMMFeatures requires: all studentt first, then lognormal, then normal.

FEATURES = [
    ('ret',    'studentt'),
    ('rv',     'lognormal')
#    ('VIX',    'lognormal'),
#    ('T10Y3M', 'normal'),
]
# ─────────────────────────────────────────────────────────────────────────────

feat_names = [f[0] for f in FEATURES]
feat_types = {f[0]: f[1] for f in FEATURES}
n_studentt  = sum(1 for _, t in FEATURES if t == 'studentt')
n_lognormal = sum(1 for _, t in FEATURES if t == 'lognormal')
n_normal    = sum(1 for _, t in FEATURES if t == 'normal')
x_dim = len(FEATURES)

print(f"Features: {feat_names}")
print(f"  studentt={n_studentt}  lognormal={n_lognormal}  normal={n_normal}  x_dim={x_dim}")

Features: ['ret', 'rv']
  studentt=1  lognormal=1  normal=0  x_dim=2


In [4]:
# temporal alignment — forward-fill all series onto SPY trading days
# (safe for series already on trading days; harmless reindex for FRED gaps)
trading_days = pool['ret'].dropna().index

aligned = pd.concat(
    [pool[name].reindex(trading_days, method='ffill').rename(name) for name in feat_names],
    axis=1
).dropna().sort_index()

print(f"Aligned dataset: {aligned.index[0].date()} → {aligned.index[-1].date()}  ({len(aligned)} rows)")
aligned.describe()

Aligned dataset: 1993-02-01 → 2026-05-15  (8380 rows)


,ret,rv
count,8380.000000,8.380000e+03
mean,0.000408,7.714262e-03
std,0.011714,6.169932e-03
min,-0.115887,1.000000e-08
25%,-0.004325,4.054098e-03
50%,0.000682,6.184425e-03
75%,0.005932,9.432116e-03
max,0.135577,8.746396e-02


In [43]:
# train / test split and normalization
#   studentt / normal  → z-score  (mean=0, std=1)
#   lognormal          → divide by training mean  (keeps support > 0, centers around 1)
K       = 15
n_train = 4500
n_test  = 1500

train_raw = aligned.iloc[:n_train]
test_raw  = aligned.iloc[n_train:n_train + n_test]

norm_params = {}
for name, etype in FEATURES:
    if etype in ('studentt', 'normal'):
        norm_params[name] = {'mean': train_raw[name].mean(), 'std': train_raw[name].std()}
    else:  # lognormal
        norm_params[name] = {'mean': train_raw[name].mean()}

def normalize(df):
    out = df.copy()
    for name, etype in FEATURES:
        p = norm_params[name]
        if etype in ('studentt', 'normal'):
            out[name] = (df[name] - p['mean']) / p['std']
        else:
            out[name] = df[name] / p['mean']
    return out

train_norm = normalize(train_raw)
test_norm  = normalize(test_raw)

print(f"train: {len(train_norm)} rows  test: {len(test_norm)} rows")
print()
train_norm.describe().round(4)

train: 4500 rows  test: 1500 rows



,ret,rv
count,4500.0000,4500.0000
mean,-0.0000,1.0000
std,1.0000,0.7833
min,-8.3349,0.0000
25%,-0.4354,0.5348
50%,0.0301,0.8212
75%,0.4653,1.2293
max,10.8479,10.1805


In [44]:
# reshape into non-overlapping sequences of length T
T = 15
train_arr   = train_norm.values.astype(np.float32)
n_sequences = len(train_arr) // T
train_data  = torch.tensor(train_arr[:n_sequences * T].reshape(n_sequences, T, x_dim))
print(f"train dataset: {train_data.shape}  ({n_sequences} sequences of length {T})")

test_data = torch.tensor(test_norm.values.astype(np.float32)).unsqueeze(0)  # (1, n_test, x_dim)
print(f"test dataset:  {test_data.shape}")

train dataset: torch.Size([300, 15, 2])  (300 sequences of length 15)
test dataset:  torch.Size([1, 1500, 2])


In [7]:
# dataloader
batch_size = 32
loader = DataLoader(TensorDataset(train_data), batch_size=batch_size, shuffle=True)
print(f"batches per epoch: {len(loader)}")

batches per epoch: 10


In [8]:
# model — emission counts derived from FEATURES config
z_dim          = 8
hidden_dim     = 64
rnn_hidden_dim = 64
rnn_num_layers = 1

dmm = DMMFeatures(
    n_studentt=n_studentt,
    n_lognormal=n_lognormal,
    n_normal=n_normal,
    z_dim=z_dim,
    hidden_dim=hidden_dim,
    rnn_hidden_dim=rnn_hidden_dim,
    rnn_num_layers=rnn_num_layers,
)

c:\Users\mikah\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\nn\modules\linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))


In [9]:
def elbo_terms(model, guide, x):
    guide_trace = poutine.trace(guide).get_trace(x)
    model_trace = poutine.trace(poutine.replay(model, trace=guide_trace)).get_trace(x)

    guide_trace.compute_log_prob()
    model_trace.compute_log_prob()

    reconstruction = sum(
        site['log_prob_sum']
        for site in model_trace.nodes.values()
        if site['type'] == 'sample' and site['is_observed']
    )
    kl = sum(
        guide_trace.nodes[name]['log_prob_sum'] - model_trace.nodes[name]['log_prob_sum']
        for name, site in guide_trace.nodes.items()
        if site['type'] == 'sample'
    )
    return reconstruction.item(), kl.item()

In [11]:
# training
pyro.clear_param_store()
optimizer = Adam({'lr': 1e-3})
svi = SVI(dmm.model, dmm.guide, optimizer, loss=Trace_ELBO())

num_epochs = 301
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for (batch,) in loader:
        epoch_loss += svi.step(batch)
    if epoch % 50 == 0:
        with torch.no_grad():
            recon, kl = elbo_terms(dmm.model, dmm.guide, train_data)
        print(f"epoch {epoch:4d}  loss: {epoch_loss / len(loader):10.4f}  recon: {recon:10.4f}  kl: {kl:10.4f}")

epoch    0  loss:  2233.4490  recon: -17479.6973  kl:  2875.7620
epoch   50  loss:  1149.6504  recon: -10376.1270  kl:   884.0745
epoch  100  loss:  1024.4598  recon: -9170.5117  kl:  1159.0720
epoch  150  loss:   959.2898  recon: -8070.6938  kl:  1426.2190
epoch  200  loss:   892.5611  recon: -7230.9453  kl:  1531.3774
epoch  250  loss:   847.6687  recon: -6912.2808  kl:  1615.4609
epoch  300  loss:   811.4599  recon: -6603.2051  kl:  1497.0342


In [12]:
# helper: fit baseline distributions on training data and evaluate on a given split
# stores fitted params so they can be reused for both train and test evaluation
base_params = {}
for name, etype in FEATURES:
    train_vals = train_norm[name].values.astype(np.float32)
    if etype == 'studentt':
        df_f, mu_f, sc_f = scipy_t.fit(train_vals)
        base_params[name] = ('studentt', df_f, mu_f, sc_f)
    elif etype == 'lognormal':
        log_vals = np.log(train_vals)
        base_params[name] = ('lognormal', float(log_vals.mean()), float(log_vals.std()))
    else:
        base_params[name] = ('normal', float(train_vals.mean()), float(train_vals.std()))

def baseline_nll(norm_df, offset):
    nlls = {}
    for name, etype in FEATURES:
        pred = torch.tensor(norm_df[name].values.astype(np.float32))[offset:]
        p = base_params[name]
        if etype == 'studentt':
            nll = -dist.StudentT(torch.tensor(p[1]), torch.tensor(p[2]), torch.tensor(p[3])).log_prob(pred).mean().item()
        elif etype == 'lognormal':
            nll = -dist.LogNormal(torch.tensor(p[1]), torch.tensor(p[2])).log_prob(pred).mean().item()
        else:
            nll = -dist.Normal(torch.tensor(p[1]), torch.tensor(p[2])).log_prob(pred).mean().item()
        nlls[name] = nll
    return nlls

def print_eval_table(base_nlls, model_breakdown, title):
    W = 12
    print(f"\n{title}")
    print(f"{'feature':<10} {'type':<12} {'baseline':>{W}} {'direct':>{W}} {'transition':>{W}}")
    print('-' * (10 + 12 + W * 3 + 6))
    for name, etype in FEATURES:
        print(f"  {name:<8} {etype:<12} {base_nlls[name]:>{W}.4f}")
    print('-' * (10 + 12 + W * 3 + 6))
    for gname in ('studentt', 'lognormal', 'normal'):
        members = [n for n, t in FEATURES if t == gname]
        if not members:
            continue
        label = f"[{', '.join(members)}]"
        group_base = sum(base_nlls[n] for n in members)
        print(f"  {label:<20} {group_base:>{W}.4f} {model_breakdown['direct'][gname]:>{W}.4f} {model_breakdown['transition'][gname]:>{W}.4f}")
    print('-' * (10 + 12 + W * 3 + 6))
    total_base = sum(base_nlls.values())
    print(f"  {'total':<20} {total_base:>{W}.4f} {model_breakdown['direct']['total']:>{W}.4f} {model_breakdown['transition']['total']:>{W}.4f}")

In [41]:
# test set evaluation
with torch.no_grad():
    test_breakdown = {
        'direct':     dmm.predict_nll_breakdown(test_data, K=K, method='direct'),
        'transition': dmm.predict_nll_breakdown(test_data, K=K, method='transition'),
    }

print_eval_table(baseline_nll(test_norm, K), test_breakdown, 'Test set')


Test set
feature    type             baseline       direct   transition
----------------------------------------------------------------
  ret      studentt           0.9585
  rv       lognormal          0.9290
----------------------------------------------------------------
  [ret]                      0.9585       0.9225       0.9167
  [rv]                       0.9290       0.3097       0.2726
----------------------------------------------------------------
  total                      1.8875       1.2321       1.1894


In [18]:
# training set evaluation
train_data_seq = torch.tensor(train_norm.values.astype(np.float32)).unsqueeze(0)  # (1, n_train, x_dim)

with torch.no_grad():
    train_breakdown = {
        'direct':     dmm.predict_nll_breakdown(train_data_seq, K=K, method='direct'),
        'transition': dmm.predict_nll_breakdown(train_data_seq, K=K, method='transition'),
    }

print_eval_table(baseline_nll(train_norm, K), train_breakdown, 'Training set')


Training set
feature    type             baseline       direct   transition
----------------------------------------------------------------
  ret      studentt           1.2919
  rv       lognormal          1.4966
----------------------------------------------------------------
  [ret]                      1.2919       1.2211       1.2175
  [rv]                       1.4966       3.2519       3.3283
----------------------------------------------------------------
  total                      2.7885       4.4730       4.5457


In [15]:
# sigma diagnostics: does the model learn time-varying scale or collapse to a constant?
# a high sigma_std means the model is actively modulating uncertainty over time
with torch.no_grad():
    z_means, _ = dmm.infer(test_data, K)

sigmas = {name: [] for name, _ in FEATURES}

with torch.no_grad():
    for t in range(z_means.shape[1]):
        df, st_mu, st_sigma, ln_mu, ln_sigma, n_mu, n_sigma = dmm.emitter(z_means[:, t, :])
        st_idx = ln_idx = n_idx = 0
        for name, etype in FEATURES:
            if etype == 'studentt':
                sigmas[name].append(st_sigma[:, st_idx].mean().item())
                st_idx += 1
            elif etype == 'lognormal':
                sigmas[name].append(ln_sigma[:, ln_idx].mean().item())
                ln_idx += 1
            else:
                sigmas[name].append(n_sigma[:, n_idx].mean().item())
                n_idx += 1

# learned df for Student-t features (global parameter, not time-varying)
df_learned = 2.0 + dmm.emitter.softplus(dmm.emitter.log_df)
for i, (name, etype) in enumerate([(n, t) for n, t in FEATURES if t == 'studentt']):
    print(f"Learned df for {name}: {df_learned[i].item():.4f}")
print()

print(f"{'feature':<10} {'mean':>10} {'std':>12} {'min':>10} {'max':>10}")
print('-' * 54)
for name, _ in FEATURES:
    s = np.array(sigmas[name])
    print(f"{name:<10} {s.mean():>10.4f} {s.std():>12.6f} {s.min():>10.4f} {s.max():>10.4f}")

Learned df for ret: 4.7094

feature          mean          std        min        max
------------------------------------------------------
ret            0.6712     0.377573     0.3052     2.3229
rv             0.3743     0.102046     0.3348     1.2514


In [40]:
from arch import arch_model
from scipy.stats import t as scipy_t_dist

# ── GARCH(1,1) with Student-t innovations ─────────────────────────────────
# Fit on normalized returns so NLL is directly comparable to DMM and static
# baselines above (all evaluated in the z-scored return space).

ret_train_z = train_norm['ret'].values.astype(np.float64)
ret_test_z  = test_norm['ret'].values.astype(np.float64)

# ── fit ────────────────────────────────────────────────────────────────────
am        = arch_model(ret_train_z, mean='Zero', vol='GARCH', p=1, q=1, dist='t')
garch_res = am.fit(disp='off', options={'ftol': 1e-9, 'maxiter': 500})
print(garch_res.summary())

omega = float(garch_res.params['omega'])
alpha = float(garch_res.params['alpha[1]'])
beta  = float(garch_res.params['beta[1]'])
nu    = float(garch_res.params['nu'])

print(f"\nomega={omega:.6f}  alpha={alpha:.4f}  beta={beta:.4f}  nu={nu:.4f}")
print(f"persistence (alpha + beta) = {alpha + beta:.6f}")

# ── filter: compute one-step-ahead conditional std dev ────────────────────
def garch11_sigma(returns, omega, alpha, beta, h0=None):
    """GARCH(1,1) variance filter.  h0: initial variance (defaults to unconditional)."""
    n    = len(returns)
    h    = np.empty(n)
    h[0] = h0 if h0 is not None else omega / (1.0 - alpha - beta)
    for t in range(1, n):
        h[t] = omega + alpha * returns[t - 1] ** 2 + beta * h[t - 1]
    return np.sqrt(h)

sigma_train = garch11_sigma(ret_train_z, omega, alpha, beta)

# warm-start test filter using the last training state
h_last    = sigma_train[-1] ** 2
h_warmup  = omega + alpha * ret_train_z[-1] ** 2 + beta * h_last
sigma_test = garch11_sigma(ret_test_z, omega, alpha, beta, h0=h_warmup)

# ── NLL ───────────────────────────────────────────────────────────────────
# arch parametrizes the Student-t innovation with unit variance:
#   r_t = sigma_t * z_t,  z_t ~ t(nu, 0, sqrt((nu-2)/nu))
#   => r_t ~ t(nu, 0, sigma_t * sqrt((nu-2)/nu))
def garch_nll(returns, sigma, nu, offset=0):
    r  = returns[offset:]
    s  = sigma[offset:] * np.sqrt((nu - 2.0) / nu)
    lp = scipy_t_dist.logpdf(r, df=nu, loc=0.0, scale=s)
    return -np.mean(lp)

# skip first K obs to match the DMM evaluation window
garch_nll_train = garch_nll(ret_train_z, sigma_train, nu, offset=K)
garch_nll_test  = garch_nll(ret_test_z,  sigma_test,  nu, offset=K)

static_nll_train = baseline_nll(train_norm, K)['ret']
static_nll_test  = baseline_nll(test_norm,  K)['ret']

print(f"\n{'model':<24} {'train NLL':>12} {'test NLL':>12}")
print('-' * 50)
print(f"  {'Static Student-t':<22} {static_nll_train:>12.4f} {static_nll_test:>12.4f}")
print(f"  {'GARCH(1,1) + t':<22} {garch_nll_train:>12.4f} {garch_nll_test:>12.4f}")

                          Zero Mean - GARCH Model Results                           
Dep. Variable:                            y   R-squared:                       0.000
Mean Model:                       Zero Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -5299.16
Distribution:      Standardized Student's t   AIC:                           10606.3
Method:                  Maximum Likelihood   BIC:                           10632.0
                                              No. Observations:                 4500
Date:                      Mon, May 18 2026   Df Residuals:                     4500
Time:                              19:10:07   Df Model:                            0
                              Volatility Model                              
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
omeg

In [45]:
from scipy.stats import lognorm as scipy_lognorm

# ── AR(1) on log-RV baseline ───────────────────────────────────────────────
# Model: log(rv_norm_t) = c + φ·log(rv_norm_{t-1}) + ε_t,  ε_t ~ N(0, σ²)
# => rv_norm_t | past ~ LogNormal(c + φ·log(rv_{t-1}), σ)
# NLL is directly comparable to the DMM's lognormal emission.

rv_train_z   = train_norm['rv'].values.astype(np.float64)
rv_test_z    = test_norm['rv'].values.astype(np.float64)
log_rv_train = np.log(rv_train_z.clip(min=1e-10))
log_rv_test  = np.log(rv_test_z.clip(min=1e-10))

# ── OLS fit on training pairs (log_rv_{t-1}, log_rv_t) ────────────────────
y    = log_rv_train[1:]
X    = np.column_stack([np.ones(len(y)), log_rv_train[:-1]])
coef = np.linalg.lstsq(X, y, rcond=None)[0]
c_ar, phi = float(coef[0]), float(coef[1])
resid_train = y - (c_ar + phi * log_rv_train[:-1])
sigma_ar    = float(resid_train.std(ddof=2))

print(f"AR(1) on log-rv:  c={c_ar:.4f}  phi={phi:.4f}  sigma={sigma_ar:.6f}")

# ── one-step-ahead NLL ─────────────────────────────────────────────────────
def ar1_lognorm_nll(rv, log_rv_lag, c, phi, sigma, offset=0):
    mu_t = c + phi * log_rv_lag
    lp   = scipy_lognorm.logpdf(rv, s=sigma, scale=np.exp(mu_t))
    return -np.mean(lp[offset:])

# training: predictions for t=1..n-1 using lag t-1; skip first K
ar1_nll_train = ar1_lognorm_nll(
    rv_train_z[1:], log_rv_train[:-1], c_ar, phi, sigma_ar, offset=K
)

# test: lag for t=0 is the last training value
log_rv_lag_test = np.concatenate([[log_rv_train[-1]], log_rv_test[:-1]])
ar1_nll_test = ar1_lognorm_nll(
    rv_test_z, log_rv_lag_test, c_ar, phi, sigma_ar, offset=K
)

static_rv_nll_train = baseline_nll(train_norm, K)['rv']
static_rv_nll_test  = baseline_nll(test_norm,  K)['rv']

print(f"\n{'model':<24} {'train NLL':>12} {'test NLL':>12}")
print('-' * 50)
print(f"  {'Static LogNormal':<22} {static_rv_nll_train:>12.4f} {static_rv_nll_test:>12.4f}")
print(f"  {'AR(1) + LogNormal':<22} {ar1_nll_train:>12.4f} {ar1_nll_test:>12.4f}")

AR(1) on log-rv:  c=-0.2742  phi=0.1810  sigma=1.491548

model                       train NLL     test NLL
--------------------------------------------------
  Static LogNormal             1.4966       0.9522
  AR(1) + LogNormal            1.4788       0.9198


In [46]:
from scipy.stats import lognorm as scipy_lognorm

# ── HAR-RV(1,5,22) vs AR(1) — fitted and evaluated in raw RV units ────────
# HAR-RV: log(rv_t) = c + β_d·log(rv_{t-1})
#                       + β_w·mean(log(rv_{t-1:t-5}))
#                       + β_m·mean(log(rv_{t-1:t-22})) + ε,  ε ~ N(0,σ²)
# AR(1):  log(rv_t) = c + φ·log(rv_{t-1})              + ε,  ε ~ N(0,σ²)
# => rv_t | past ~ LogNormal(predicted_mu_t, σ) for both

rv_train_raw     = train_raw['rv'].values.astype(np.float64)
rv_test_raw      = test_raw['rv'].values.astype(np.float64)
log_rv_train_raw = np.log(rv_train_raw.clip(min=1e-10))
log_rv_test_raw  = np.log(rv_test_raw.clip(min=1e-10))

MAX_LAG = 22

# ── OLS fit ────────────────────────────────────────────────────────────────
def make_har_X(log_rv, start):
    rows = [[1.0, log_rv[t-1], log_rv[t-5:t].mean(), log_rv[t-22:t].mean()]
            for t in range(start, len(log_rv))]
    return np.array(rows)

# HAR-RV
X_har    = make_har_X(log_rv_train_raw, MAX_LAG)
y_har    = log_rv_train_raw[MAX_LAG:]
coef_har = np.linalg.lstsq(X_har, y_har, rcond=None)[0]
c_har, beta_d, beta_w, beta_m = coef_har
sigma_har = float((y_har - X_har @ coef_har).std(ddof=4))

# AR(1) on raw log-rv
X_ar1    = np.column_stack([np.ones(len(log_rv_train_raw) - 1), log_rv_train_raw[:-1]])
y_ar1    = log_rv_train_raw[1:]
coef_ar1 = np.linalg.lstsq(X_ar1, y_ar1, rcond=None)[0]
c_ar1, phi_ar1 = float(coef_ar1[0]), float(coef_ar1[1])
sigma_ar1 = float((y_ar1 - X_ar1 @ coef_ar1).std(ddof=2))

print(f"AR(1):  c={c_ar1:.4f}  phi={phi_ar1:.4f}  sigma={sigma_ar1:.6f}")
print(f"HAR-RV: c={c_har:.4f}  β_d={beta_d:.4f}  β_w={beta_w:.4f}  β_m={beta_m:.4f}  sigma={sigma_har:.6f}")

# ── one-step-ahead NLL ─────────────────────────────────────────────────────
def lognorm_nll(rv, mu_t, sigma, offset=0):
    lp = scipy_lognorm.logpdf(rv[offset:], s=sigma, scale=np.exp(mu_t[offset:]))
    return -np.mean(lp)

# training
mu_har_train = X_har @ coef_har                              # for rv[MAX_LAG:]
mu_ar1_train = c_ar1 + phi_ar1 * log_rv_train_raw[:-1]      # for rv[1:]

# offset relative to each model's first valid prediction
nll_har_train = lognorm_nll(rv_train_raw[MAX_LAG:], mu_har_train, sigma_har, offset=max(0, K - MAX_LAG))
nll_ar1_train = lognorm_nll(rv_train_raw[1:],       mu_ar1_train, sigma_ar1, offset=K)

# test: use full series so HAR can look back into training history
log_rv_full = np.concatenate([log_rv_train_raw, log_rv_test_raw])
n_tr        = len(log_rv_train_raw)

X_har_test   = make_har_X(log_rv_full, start=n_tr)           # n_test rows
mu_har_test  = X_har_test @ coef_har

log_rv_lag_test = np.concatenate([[log_rv_train_raw[-1]], log_rv_test_raw[:-1]])
mu_ar1_test     = c_ar1 + phi_ar1 * log_rv_lag_test

nll_har_test = lognorm_nll(rv_test_raw, mu_har_test,  sigma_har, offset=K)
nll_ar1_test = lognorm_nll(rv_test_raw, mu_ar1_test,  sigma_ar1, offset=K)

print(f"\n{'model':<24} {'train NLL':>12} {'test NLL':>12}  (raw RV units)")
print('-' * 54)
print(f"  {'AR(1) + LogNormal':<22} {nll_ar1_train:>12.4f} {nll_ar1_test:>12.4f}")
print(f"  {'HAR-RV + LogNormal':<22} {nll_har_train:>12.4f} {nll_har_test:>12.4f}")


AR(1):  c=-4.1702  phi=0.1810  sigma=1.491548
HAR-RV: c=-0.9536  β_d=0.0129  β_w=-0.0206  β_m=0.8195  sigma=1.377260

model                       train NLL     test NLL  (raw RV units)
------------------------------------------------------
  AR(1) + LogNormal           -3.2782      -3.8372
  HAR-RV + LogNormal          -3.3468      -3.9173


In [48]:
log_rv = np.log(rv_train_z)
print(f"min log_rv: {log_rv.min():.4f}")
print(f"n observations below -5: {(log_rv < -5).sum()}")

min log_rv: -13.6637
n observations below -5: 49
